# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 05: Pydantic v2 / DTOs en PHP y Tipado Estático
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *DTO manual con validación de tipos primitivos*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Esquema Pydantic v2 con validadores personalizados* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de tipos incompatibles y coerción*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion05.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** DTO manual con validación de tipos primitivos

In [7]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, Request, status

app = FastAPI(title="API Artesano")

class ManualDTO:
    def __init__(self, payload: dict):
        if not isinstance(payload, dict):
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_CONTENT,
                detail="El cuerpo debe ser un objeto JSON"
            )

        dato = payload.get("dato")
        if not isinstance(dato, str) or not dato.strip():
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_CONTENT,
                detail="El campo 'dato' debe ser una cadena (str) no vacía"
            )

        cantidad = payload.get("cantidad", 1)
        if isinstance(cantidad, bool) or not isinstance(cantidad, int):
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_CONTENT,
                detail="El campo 'cantidad' debe ser un entero (int) estricto"
            )

        self.dato = dato.strip()
        self.cantidad = cantidad

@app.post("/api/recurso")
async def procesar_recurso(request: Request):
    try:
        body = await request.json()
    except Exception:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="JSON malformado")

    dto = ManualDTO(body)
    return {
        "status": "success",
        "mensaje": f"Procesado: {dto.dato}",
        "cantidad": dto.cantidad
    }

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de tipos incompatibles y coerción

In [8]:
%%writefile app_ia.py
from fastapi import FastAPI
from pydantic import BaseModel, Field, StrictInt, StrictStr, field_validator

app = FastAPI(title="API Cientifico - Pydantic v2")

class RecursoSchema(BaseModel):
    dato: StrictStr = Field(..., min_length=1)
    cantidad: StrictInt = Field(default=1, gt=0)

    @field_validator("dato")
    @classmethod
    def validar_dato_no_vacio(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El campo 'dato' no puede contener solo espacios")
        return v.strip()

@app.post("/api/recurso")
def procesar_recurso(payload: RecursoSchema):
    return {
        "status": "success",
        "mensaje": f"Procesado: {payload.dato}",
        "cantidad": payload.cantidad
    }

Overwriting app_ia.py


In [6]:
import threading
import time
import uvicorn
import httpx
import sys
import importlib

# Forzar la recarga del módulo por si hubo cambios previos en app_artesano
import app_artesano
importlib.reload(app_artesano)
from app_artesano import app

def run_server():
    try:
        uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
    except Exception as e:
        print(f"Nota del Servidor: {e}")

# Levantamos el servidor en un hilo secundario si no está activo
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2.0)  # Damos suficiente tiempo para que el puerto se asocie

print("=== AUDITORIA CRITICA WEB: SESION 05 ===\n")

with httpx.Client(base_url="http://127.0.0.1:8000") as client:
    try:
        # Caso exitoso
        r = client.post("/api/recurso", json={"dato": "Prueba Test Harness", "cantidad": 5})
        print(f"Respuesta exitosa: HTTP {r.status_code} - {r.json()}")
        assert r.status_code == 200

        # Caso de error esperado (Dato inválido)
        r_err = client.post("/api/recurso", json={"dato": "", "cantidad": 5})
        print(f"Respuesta error validación: HTTP {r_err.status_code} - {r_err.json()}")
        assert r_err.status_code == 422

        print("\nPASO ✅ - Test Harness HTTP ejecutado con éxito total.")
    except Exception as e:
        print(f"Error de conexión en el test harness: {e}")

=== AUDITORIA CRITICA WEB: SESION 05 ===

Respuesta exitosa: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness', 'cantidad': 5}
Respuesta error validación: HTTP 422 - {'detail': "El campo 'dato' debe ser una cadena (str) no vacía"}

PASO ✅ - Test Harness HTTP ejecutado con éxito total.


/content/app_artesano.py:37: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  dto = ManualDTO(body)
